In [ ]:
import re
import unicodedata
import json
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import geobr
from IPython.display import display




In [ ]:
CSV_CASOS_ANO_DIAGNOSTICO = '/home/anna/tcc/Tratamento/Casos/incidencia _ano_diagnostico.csv'
CSV_CASOS_ESCOLARIDADE    = '/home/anna/tcc/Tratamento/Casos/incidencia _escolaridade.csv'
CSV_CASOS_RACA_COR        = '/home/anna/tcc/Tratamento/Casos/incidencia _raca_cor.csv'
CSV_CASOS_FAIXA_ETARIA    = '/home/anna/tcc/Tratamento/Casos/incidencia _faixa_etaria.csv'
CSV_POPULACAO_FAIXA_ETARIA = '/home/anna/tcc/Tratamento/População - Faixa Etária/tabela_populacao_faixa_etaria.csv'
MUNICIPIO = None
CODIGO_MUNICIPIO = None
REGIONAL_SAUDE = None
ANOS_RANGE_CASOS = (1985, 2023)
TOP_N_CATEGORIAS = 25
COR_BARRAS = "#6ca8e5"
COR_PONTOS = "#d73f3f"
LARGURA_GRAFICO = 2300
ALTURA_GRAFICO_CASOS = 1700
X_MIN = 0
DIVISOR_MEDIA_ANUAL_CASOS = 10
ANO_REFERENCIA_POPULACAO_CASOS = 2022
ANO_MALHA_MUNICIPIOS_MG = 2020
FILTRO_FAIXA_ETARIA = None
ORDENAR_TABELA_POR = 'valor_absoluto_casos'




In [ ]:
def _norm_txt(txt: str) -> str:
    return unicodedata.normalize('NFKD', str(txt)).encode('ascii','ignore').decode('ascii').lower().strip()
def _norm_col(col: str) -> str:
    return re.sub(r'\s+', '_', _norm_txt(col))
def _find_first_col(df: pd.DataFrame, candidates_norm: set[str]):
    for col in df.columns:
        if _norm_col(col) in candidates_norm:
            return col
    return None
def _drop_index_cols(df: pd.DataFrame) -> pd.DataFrame:
    cols_drop = [c for c in df.columns if _norm_col(c).startswith('unnamed')]
    return df.drop(columns=cols_drop) if cols_drop else df.copy()
def _fmt_int(n):
    try:
        return f'{int(n):,}'.replace(',', '.')
    except Exception:
        return str(n)
FAIXAS_ETARIAS_ORDEM = [
    '0-9',
    '10-19',
    '20-29',
    '30-39',
    '40-49',
    '50-59',
    '60-69',
    '70 ou mais',
]
def normalizar_faixa_etaria(categoria):
    cat = _norm_txt(categoria)
    if cat in {'70-79', '70 a 79', '80+','80 ou mais', '80 ou mais.', '70 ou mais'}:
        return '70 ou mais'
    return str(categoria).strip()
def _normalizar_codigo_municipio(serie: pd.Series) -> pd.Series:
    return pd.to_numeric(serie, errors='coerce').astype('Int64').astype(str).str.replace('<NA>','',regex=False).str.zfill(7)
def aplicar_filtros(df: pd.DataFrame) -> pd.DataFrame:
    out = _drop_index_cols(df)
    col_mun = _find_first_col(out, {'municipios','municipio','cidades','cidade'})
    col_cod = _find_first_col(out, {'codigo','cod'})
    col_reg = _find_first_col(out, {'regional_de_saude','regional_saude','regional'})
    if MUNICIPIO is not None and col_mun:
        out = out[out[col_mun].astype(str).map(_norm_txt) == _norm_txt(MUNICIPIO)]
    if CODIGO_MUNICIPIO is not None and col_cod:
        out = out[pd.to_numeric(out[col_cod], errors='coerce').astype('Int64') == int(CODIGO_MUNICIPIO)]
    if REGIONAL_SAUDE is not None and col_reg:
        out = out[out[col_reg].astype(str).map(_norm_txt) == _norm_txt(REGIONAL_SAUDE)]
    return out
def _year_cols(df):
    cols=[]
    for c in df.columns:
        try:
            y=int(c)
            if 1900<=y<=2100: cols.append(c)
        except: pass
    return sorted(cols)
def preparar_ano_diagnostico(df):
    ycols=_year_cols(df)
    s=df[ycols].apply(pd.to_numeric,errors='coerce').fillna(0).sum(axis=0)
    s=s[s>0]
    return pd.DataFrame({'ano':s.index.astype(int),'categoria':s.index.astype(str),'total':s.values}).sort_values('ano').reset_index(drop=True)
def preparar_long_wide(df):
    rows=[]
    for c in df.columns:
        txt=str(c)
        if '_' not in txt: continue
        ano_str,categoria=txt.rsplit('_',1)
        try: ano=int(ano_str)
        except: continue
        categoria = normalizar_faixa_etaria(categoria)
        total=pd.to_numeric(df[c],errors='coerce').fillna(0).sum()
        if total>0: rows.append({'ano':ano,'categoria':categoria,'total':int(total)})
    out=pd.DataFrame(rows)
    return out.sort_values(['ano','categoria']).reset_index(drop=True)
def build_por_categoria_por_ano(df_long, anos_range=None, top_n=None):
    d=df_long.copy()
    d['ano']=pd.to_numeric(d['ano'],errors='coerce').astype('Int64')
    d['total']=pd.to_numeric(d['total'],errors='coerce')
    d=d.dropna(subset=['ano','categoria','total'])
    d['ano']=d['ano'].astype(int)
    d=d[d['total']>0]
    if anos_range is not None:
        ai,af=map(int,anos_range)
        d=d[(d['ano']>=ai)&(d['ano']<=af)]
    por_categoria=d.groupby('categoria')['total'].sum()
    if top_n is not None: por_categoria=por_categoria.sort_values(ascending=False).head(int(top_n)).sort_values()
    por_ano=d.groupby('ano')['total'].sum().sort_index()
    return por_categoria, por_ano
def _carregar_e_preparar_casos(caminho_csv, nome, modo='wide', top_n=None, anos_range=ANOS_RANGE_CASOS):
    df=aplicar_filtros(pd.read_csv(caminho_csv))
    print(f'Linhas selecionadas ({nome}): {len(df)}')
    long=preparar_ano_diagnostico(df) if modo=='ano_cols' else preparar_long_wide(df)
    return build_por_categoria_por_ano(long, anos_range=anos_range, top_n=top_n)
print('=== CONFIG / FILTRO ===')
print('MUNICIPIO:', MUNICIPIO)
print('CODIGO_MUNICIPIO:', CODIGO_MUNICIPIO)
print('REGIONAL_SAUDE:', REGIONAL_SAUDE)
graficos_casos=[]
por_cat,por_ano=_carregar_e_preparar_casos(CSV_CASOS_ANO_DIAGNOSTICO,'Casos - Ano Diagnóstico',modo='ano_cols',top_n=None,anos_range=None)
por_ano_ref=por_ano[(por_ano.index>=ANOS_RANGE_CASOS[0])&(por_ano.index<=ANOS_RANGE_CASOS[1])]
graficos_casos.append({'tipo':'ano','titulo':'Casos - Ano Diagnóstico','ylabel':'Ano do diagnóstico','por_categoria':por_cat,'por_ano':por_ano_ref,'x_max':8000})
for nome,csv,ylabel,xmax in [
    ('Casos - Escolaridade',CSV_CASOS_ESCOLARIDADE,'Escolaridade',37000),
    ('Casos - Raça/Cor',CSV_CASOS_RACA_COR,'Raça/Cor',54000),
    ('Casos - Faixa Etária',CSV_CASOS_FAIXA_ETARIA,'Faixa etária',33000),
]:
    c,a=_carregar_e_preparar_casos(csv,nome,modo='wide',top_n=TOP_N_CATEGORIAS,anos_range=ANOS_RANGE_CASOS)
    graficos_casos.append({'tipo':'duplo','titulo':nome,'ylabel':ylabel,'por_categoria':c,'por_ano':a,'x_max':xmax})
print(f'Casos: {len(graficos_casos)} gráficos preparados')




In [ ]:
def _round_up_nice(v):
    v=float(v)
    if v<=0: return 1000
    step=200 if v<=2000 else 500 if v<=10000 else 1000
    return int(((v+step-1)//step)*step)
def _build_ticks(xmax):
    step=200 if xmax<=2000 else 500 if xmax<=10000 else 1000
    return list(range(X_MIN,int(xmax)+step,step))
EXTRA_EIXO = 1000  # novo
def _resolve_xmax(vals, fixo=None, folga=0.45, extra=0):
    d = max(vals) if vals else 0
    r = _round_up_nice(d * (1 + folga))
    base = r if fixo is None else max(int(fixo), r)
    return base + extra
def plot_ano(por_ano,titulo,xmax_fixo):
    s=pd.to_numeric(por_ano,errors='coerce').fillna(0)
    s=s[s>0]
    anos=s.index.astype(int).tolist(); vals=s.values.astype(int).tolist()
    xmax = _resolve_xmax(vals, xmax_fixo, folga=0.25, extra=EXTRA_EIXO)
    ticks = _build_ticks(xmax)
    fig=go.Figure(go.Bar(
        x=[str(a) for a in anos],
        y=vals,
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in vals],
        textposition='inside',
        insidetextanchor='end',
        textfont=dict(size=20),
        cliponaxis=False,
        constraintext='inside'
    ))
    fig.update_layout(
        title=dict(text=titulo, x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',
        margin=dict(l=170, r=220, t=90, b=80),
        xaxis=dict(
            title='Ano do diagnóstico',
            title_font=dict(size=26),   # título do eixo X
            tickfont=dict(size=18),     # valores do eixo X
            tickangle=-45,              # girar para diagonal
            showgrid=True, gridcolor='#d7dce3', gridwidth=1, automargin=True
        ),
        yaxis=dict(
            title='Número de casos',
            title_font=dict(size=32),   # título do eixo Y
            tickfont=dict(size=22),     # valores do eixo Y
            tickmode='array', tickvals=ticks, ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN, xmax], showgrid=True, gridcolor='#d7dce3', gridwidth=1, automargin=True
        ),
    )
    fig.show()
def plot_duplo(por_categoria,por_ano,titulo,ytitle,xmax_fixo):
    c=pd.to_numeric(por_categoria,errors='coerce').fillna(0); c=c[c>0]
    a=pd.to_numeric(por_ano,errors='coerce').fillna(0); a=a[a>0]
    is_faixa = _norm_txt(ytitle) == 'faixa etaria'
    if is_faixa:
        ordem = [f for f in FAIXAS_ETARIAS_ORDEM if f in c.index.astype(str).tolist()]
        c = c.reindex(ordem)
    cats=c.index.astype(str).tolist(); v1=c.values.astype(int).tolist()
    anos=a.index.astype(int).tolist(); v2=a.values.astype(int).tolist()
    xmax_base = int(xmax_fixo) if xmax_fixo is not None else _resolve_xmax(v1 + v2, None, folga=0.50)
    xmax = xmax_base + EXTRA_EIXO
    ticks=_build_ticks(xmax)
    ano_ini, ano_fim = map(int, ANOS_RANGE_CASOS)
    anos_eixo = list(range(ano_ini, ano_fim + 1))
    fig=go.Figure()
    fig.add_trace(go.Bar(
        name='Total por categoria',x=v1,y=cats,orientation='h',
        marker=dict(color=COR_BARRAS),
        text=[_fmt_int(v) for v in v1],
        textposition='inside',
        insidetextanchor='end',
        textfont=dict(size=20),
        cliponaxis=False,
        constraintext='inside',
        yaxis='y',xaxis='x'
    ))
    fig.add_trace(go.Scatter(
        name='Total por ano',x=v2,y=anos,mode='markers+text',
        marker=dict(color=COR_PONTOS,size=10),
        text=[_fmt_int(v) for v in v2],
        textfont=dict(color=COR_PONTOS, size=20),
        textposition='middle right',
        cliponaxis=False,
        yaxis='y2',xaxis='x'
    ))
    fig.update_layout(
        title=dict(text=titulo,x=0.5),
        height=ALTURA_GRAFICO_CASOS,
        width=LARGURA_GRAFICO,
        plot_bgcolor='#f2f4f7',
        paper_bgcolor='white',
        margin=dict(l=170, r=350, t=95, b=80),
        
        xaxis=dict(
            title='Número de casos',
            tickmode='array',
            tickvals=ticks,
            ticktext=[_fmt_int(v) for v in ticks],
            range=[X_MIN,xmax],
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18),
            tickangle=-90
        ),
        yaxis=dict(
            title=ytitle,
            showgrid=True,
            gridcolor='#d7dce3',
            gridwidth=1,
            categoryorder='array',
            categoryarray=cats,
            automargin=True,
            title_font=dict(size=32),
            tickfont=dict(size=22)
        ),
        yaxis2=dict(
            title='Ano',
            overlaying='y',
            side='right',
            tickmode='array',
            tickvals=anos_eixo,
            range=[ano_ini-2, ano_fim+2],
            showgrid=False,
            automargin=True,
            title_font=dict(size=26),
            tickfont=dict(size=18)
        ),
    )
    fig.show()
for g in graficos_casos:
    if g['tipo']=='ano':
        plot_ano(g['por_ano'],f"{g['titulo']} (total por ano)",g['x_max'])
    else:
        plot_duplo(g['por_categoria'],g['por_ano'],g['titulo'],g['ylabel'],g['x_max'])
print('✅ Gráficos de casos plotados')




In [ ]:
def _peso_faixa(df_segmento,filtro_faixa=None):
    base=_drop_index_cols(df_segmento)
    col_mun=_find_first_col(base,{'municipios','municipio','cidades','cidade'})
    col_cod=_find_first_col(base,{'codigo_municipio','cod_municipio','codigo','cod'})
    filtro=None if filtro_faixa is None else ({_norm_txt(v) for v in filtro_faixa} if isinstance(filtro_faixa,(list,tuple,set)) else {_norm_txt(filtro_faixa)})
    rows=[]
    for col in base.columns:
        txt=str(col)
        if '_' not in txt: continue
        ano_str,cat=txt.split('_',1)
        if not ano_str.isdigit(): continue
        rows.append(pd.DataFrame({'codigo_municipio':_normalizar_codigo_municipio(base[col_cod]),'nome_municipio':base[col_mun].astype(str).str.strip(),'ano':int(ano_str),'valor':pd.to_numeric(base[col],errors='coerce').fillna(0),'sel':True if filtro is None else _norm_txt(cat) in filtro}))
    long=pd.concat(rows,ignore_index=True)
    tot=long.groupby(['codigo_municipio','nome_municipio','ano'],as_index=False)['valor'].sum()
    sel=long[long['sel']].groupby(['codigo_municipio','nome_municipio','ano'],as_index=False)['valor'].sum().rename(columns={'valor':'valor_sel'})
    out=tot.merge(sel,on=['codigo_municipio','nome_municipio','ano'],how='left')
    out['valor_sel']=pd.to_numeric(out['valor_sel'],errors='coerce').fillna(0)
    out['peso_faixa']=1.0 if filtro is None else out['valor_sel'].div(out['valor'].replace(0,pd.NA)).fillna(0.0)
    return out[['codigo_municipio','nome_municipio','ano','peso_faixa']]
# base anual principal
df_ano=aplicar_filtros(pd.read_csv(CSV_CASOS_ANO_DIAGNOSTICO))
col_mun=_find_first_col(df_ano,{'municipios','municipio','cidades','cidade'})
col_cod=_find_first_col(df_ano,{'codigo_municipio','cod_municipio','codigo','cod'})
anos=[c for c in df_ano.columns if str(c).isdigit()]
rows=[]
for c in anos:
    rows.append(pd.DataFrame({'codigo_municipio':_normalizar_codigo_municipio(df_ano[col_cod]),'nome_municipio':df_ano[col_mun].astype(str).str.strip(),'ano':int(c),'total_casos_ano':pd.to_numeric(df_ano[c],errors='coerce').fillna(0)}))
base_ano=pd.concat(rows,ignore_index=True).groupby(['codigo_municipio','nome_municipio','ano'],as_index=False)['total_casos_ano'].sum()
peso=_peso_faixa(pd.read_csv(CSV_CASOS_FAIXA_ETARIA),FILTRO_FAIXA_ETARIA)
casos_filtrados=base_ano.merge(peso,on=['codigo_municipio','nome_municipio','ano'],how='left')
casos_filtrados['peso_faixa']=pd.to_numeric(casos_filtrados['peso_faixa'],errors='coerce').fillna(1.0)
casos_filtrados['casos_ajustados_ano']=casos_filtrados['total_casos_ano']*casos_filtrados['peso_faixa']
municipios=casos_filtrados.groupby(['codigo_municipio','nome_municipio'],as_index=False).agg(valor_absoluto_casos=('casos_ajustados_ano','sum'),variancia_anual=('casos_ajustados_ano',lambda s: float(pd.Series(s).var(ddof=0))),desvio_padrao_anual=('casos_ajustados_ano',lambda s: float(pd.Series(s).std(ddof=0))))
municipios['media_anual_casos']=municipios['valor_absoluto_casos']/float(DIVISOR_MEDIA_ANUAL_CASOS)
# população do ano de referência
pop=_drop_index_cols(pd.read_csv(CSV_POPULACAO_FAIXA_ETARIA))
pm=_find_first_col(pop,{'municipios','municipio','cidades','cidade'})
pc=_find_first_col(pop,{'codigo_municipio','cod_municipio','codigo','cod'})
pcols=[c for c in pop.columns if str(c).startswith(f'{ANO_REFERENCIA_POPULACAO_CASOS}_')]
pop['populacao_ano_referencia']=pop[pcols].apply(pd.to_numeric,errors='coerce').fillna(0).sum(axis=1)
pop_ref=pop.assign(codigo_municipio=_normalizar_codigo_municipio(pop[pc]),nome_municipio=pop[pm].astype(str).str.strip()).groupby(['codigo_municipio','nome_municipio'],as_index=False)['populacao_ano_referencia'].sum()
consolidado=municipios.merge(pop_ref[['codigo_municipio','populacao_ano_referencia']],on='codigo_municipio',how='left')
consolidado['populacao_ano_referencia']=pd.to_numeric(consolidado['populacao_ano_referencia'],errors='coerce').fillna(0)
consolidado['percentual_casos_pop']=(consolidado['valor_absoluto_casos'].div(consolidado['populacao_ano_referencia'].replace(0,pd.NA)).fillna(0)*100)
def _classe_1a10(s):
    bins=[-float('inf'),0,1,2,3,4,5,6,7,8,9,float('inf')]
    labels=['0','1','2','3','4','5','6','7','8','9','10+']
    return pd.cut(pd.to_numeric(s,errors='coerce').fillna(0),bins=bins,labels=labels,right=True).astype(str)
consolidado['escala_media_1a10+']=_classe_1a10(consolidado['media_anual_casos'])
consolidado['escala_percentual_1a10+']=_classe_1a10(consolidado['percentual_casos_pop'])
mg=geobr.read_municipality(code_muni='MG',year=ANO_MALHA_MUNICIPIOS_MG).to_crs(4326)
mg['codigo_municipio']=_normalizar_codigo_municipio(mg['code_muni'])
geojson_mg=json.loads(mg[['codigo_municipio','geometry']].to_json())
mapa=mg[['codigo_municipio']].merge(consolidado,on='codigo_municipio',how='left')
cores={'0':'#f7fbff','1':'#deebf7','2':'#c6dbef','3':'#9ecae1','4':'#6baed6','5':'#4292c6','6':'#2171b5','7':'#08519c','8':'#fcbba1','9':'#fb6a4a','10+':'#cb181d'}
def _plot(df,col_val,col_classe,titulo,legenda):
    fig=px.choropleth(df,geojson=geojson_mg,locations='codigo_municipio',featureidkey='properties.codigo_municipio',color=col_classe,color_discrete_map=cores,category_orders={col_classe:['0','1','2','3','4','5','6','7','8','9','10+']},hover_name='nome_municipio',hover_data={'codigo_municipio':True,col_val:':.2f','valor_absoluto_casos':':.2f','percentual_casos_pop':':.2f','media_anual_casos':':.2f'},labels={col_classe:legenda},title=titulo)
    fig.update_geos(fitbounds='locations',visible=False,showcountries=False,showcoastlines=False,showland=False,showframe=False)
    fig.update_layout(template='plotly_white',width=1250,height=820,margin=dict(l=10,r=10,t=60,b=10),title=dict(x=0.5,xanchor='center'))
    display(fig)
# _plot(mapa,'media_anual_casos','escala_media_1a10+','MG — Média anual de casos por município (faixa etária)','Escala (1 a 10+) - média anual')
# _plot(mapa,'percentual_casos_pop','escala_percentual_1a10+',f'MG — Percentual de casos em relação à população ({ANO_REFERENCIA_POPULACAO_CASOS}) (faixa etária)','Escala (1 a 10+) - percentual')
# tabela no design DataFrame
tabela_final=consolidado[['codigo_municipio','nome_municipio','valor_absoluto_casos','media_anual_casos','variancia_anual','desvio_padrao_anual','populacao_ano_referencia','percentual_casos_pop']].copy()
if ORDENAR_TABELA_POR in tabela_final.columns:
    tabela_final=tabela_final.sort_values(ORDENAR_TABELA_POR,ascending=False).reset_index(drop=True)
tabela_design=tabela_final.rename(columns={'codigo_municipio':'Código IBGE','nome_municipio':'Município','valor_absoluto_casos':'Valor absoluto (casos)','media_anual_casos':'Média anual (casos)','variancia_anual':'Variância anual','desvio_padrao_anual':'Desvio padrão anual','populacao_ano_referencia':f'População ({ANO_REFERENCIA_POPULACAO_CASOS})','percentual_casos_pop':'Percentual casos/população (%)'})
for col in ['Valor absoluto (casos)','Média anual (casos)','Variância anual','Desvio padrão anual','Percentual casos/população (%)']:
    tabela_design[col]=pd.to_numeric(tabela_design[col],errors='coerce').round(2)
tabela_design[f'População ({ANO_REFERENCIA_POPULACAO_CASOS})']=pd.to_numeric(tabela_design[f'População ({ANO_REFERENCIA_POPULACAO_CASOS})'],errors='coerce').fillna(0).astype(int)
display(tabela_design)
print('✅ Mapas MG e tabela de incidência gerados')




In [ ]:
classificacoes_casos = [
    ("Ano diagnóstico", CSV_CASOS_ANO_DIAGNOSTICO, "ano_cols"),
    ("Escolaridade", CSV_CASOS_ESCOLARIDADE, "wide"),
    ("Faixa etária", CSV_CASOS_FAIXA_ETARIA, "wide"),
    ("Raça/Cor", CSV_CASOS_RACA_COR, "wide"),
]
resumo_classificacao = []
detalhe_categorias = []
resultados_brutos = []
total_acumulado_anos = 0.0
for nome, caminho_csv, modo in classificacoes_casos:
    por_categoria, por_ano = _carregar_e_preparar_casos(
        caminho_csv,
        nome,
        modo=modo,
        top_n=None,
        anos_range=ANOS_RANGE_CASOS,
    )
    total_classificacao = float(pd.to_numeric(por_ano, errors="coerce").fillna(0).sum())
    if nome == "Ano diagnóstico":
        total_acumulado_anos = total_classificacao
    resultados_brutos.append(
        {
            "nome": nome,
            "total": total_classificacao,
            "por_categoria": pd.to_numeric(por_categoria, errors="coerce").fillna(0),
        }
    )
for item in resultados_brutos:
    nome = item["nome"]
    total_original = item["total"]
    serie_categoria = item["por_categoria"]
    fator_ajuste = (total_acumulado_anos / total_original) if total_original else 0.0
    serie_ajustada = serie_categoria * fator_ajuste
    total_ajustado = float(serie_ajustada.sum())
    resumo_classificacao.append(
        {
            "Classificação": nome,
            "Total de casos": total_ajustado,
            "% do total acumulado por anos": (
                (total_ajustado / total_acumulado_anos) * 100 if total_acumulado_anos else 0.0
            ),
        }
    )
    for categoria, valor in serie_ajustada.items():
        percentual_na_classificacao = (
            (float(valor) / total_ajustado) * 100 if total_ajustado else 0.0
        )
        detalhe_categorias.append(
            {
                "Classificação": nome,
                "Categoria": str(categoria),
                "Total de casos": float(valor),
                "% na classificação": percentual_na_classificacao,
            }
        )
# Tabela 1: total por classificação
tabela_classificacao = pd.DataFrame(resumo_classificacao)
tabela_classificacao["Total de casos"] = (
    pd.to_numeric(tabela_classificacao["Total de casos"], errors="coerce").fillna(0).round(0).astype(int)
)
tabela_classificacao["% do total acumulado por anos"] = (
    pd.to_numeric(tabela_classificacao["% do total acumulado por anos"], errors="coerce").fillna(0).round(2)
)
# Tabela 2: detalhe por categoria dentro de cada classificação
tabela_categorias = pd.DataFrame(detalhe_categorias)
tabela_categorias["Total de casos"] = (
    pd.to_numeric(tabela_categorias["Total de casos"], errors="coerce").fillna(0).round(0).astype(int)
)
tabela_categorias["% na classificação"] = (
    pd.to_numeric(tabela_categorias["% na classificação"], errors="coerce").fillna(0).round(2)
)
tabela_categorias = tabela_categorias.sort_values(
    ["Classificação", "% na classificação", "Total de casos"],
    ascending=[True, False, False]
).reset_index(drop=True)
print(
    f"Total acumulado por anos (Ano diagnóstico, {ANOS_RANGE_CASOS[0]}–{ANOS_RANGE_CASOS[1]}): "
    f"{_fmt_int(total_acumulado_anos)}"
)
print("\nResumo por classificação (normalizado):")
display(tabela_classificacao)
print("\nDetalhe por categoria (% dentro da própria classificação):")
display(tabela_categorias)
print("✅ Tabelas de classificação e categorias geradas")




In [ ]:
# Mapa MG (municípios): casos acumulados por faixa etária (todas as faixas + todos os anos)
import json
import re
import unicodedata
from pathlib import Path
import pandas as pd
import plotly.express as px
import geobr
CSV_CASOS_FAIXA_ETARIA_MAPA = Path('/home/anna/tcc/Tratamento/Casos/incidencia _faixa_etaria.csv')
CSV_MUNICIPIOS_REGIONAIS = Path('/home/anna/tcc/Tratamento/Municípios/tabela_municipios_regionais.csv')
ANO_MALHA_MUNICIPIOS_MG_MAPA = 2020
INTERVALOS_MANUAIS_MAPA = [0, 50, 100, 500, 1000, 5000, 23000]
N_INTERVALOS_MAPA = len(INTERVALOS_MANUAIS_MAPA) - 1
COR_SEM_CASOS = '#6ca8e5'
ROTULO_SEM_CASOS = 'Sem casos (0)'
def _norm_txt_mapa(txt: str) -> str:
    t = unicodedata.normalize('NFKD', str(txt))
    t = ''.join(c for c in t if not unicodedata.combining(c))
    return t.lower().strip()
def _norm_col_mapa(col: str) -> str:
    return re.sub(r'\s+', '_', _norm_txt_mapa(col))
def _find_first_col_mapa(df: pd.DataFrame, candidatos_norm: set[str]):
    for coluna in df.columns:
        if _norm_col_mapa(coluna) in candidatos_norm:
            return coluna
    return None
def _normalizar_codigo_municipio_mapa(serie: pd.Series) -> pd.Series:
    return (
        pd.to_numeric(serie, errors='coerce')
        .astype('Int64')
        .astype(str)
        .str.replace('<NA>', '', regex=False)
        .str.zfill(7)
    )
if not CSV_CASOS_FAIXA_ETARIA_MAPA.exists():
    raise FileNotFoundError(f'Arquivo não encontrado: {CSV_CASOS_FAIXA_ETARIA_MAPA}')
base = pd.read_csv(CSV_CASOS_FAIXA_ETARIA_MAPA)
col_mun = _find_first_col_mapa(base, {'municipios', 'municipio', 'cidades', 'cidade'})
col_cod = _find_first_col_mapa(base, {'codigo_municipio', 'cod_municipio', 'codigo', 'cod'})
col_reg = _find_first_col_mapa(base, {'regional', 'regional_saude', 'urs', 'grs', 'macro', 'macrorregiao'})
if col_mun is None or col_cod is None:
    raise KeyError('Não foi possível identificar as colunas de município e código no CSV de faixa etária.')
colunas_faixa_ano = [
    c for c in base.columns
    if re.match(r'^\d{4}_.+', str(c).strip())
]
if not colunas_faixa_ano:
    raise KeyError('Não encontrei colunas no formato ANO_FAIXA_ETARIA para calcular o acumulado.')
base[colunas_faixa_ano] = base[colunas_faixa_ano].apply(pd.to_numeric, errors='coerce').fillna(0)
group_cols = ['codigo_municipio', 'nome_municipio']
if col_reg is not None:
    group_cols.append('regional')
acumulado_municipio = (
    base.assign(
        codigo_municipio=_normalizar_codigo_municipio_mapa(base[col_cod]),
        nome_municipio=base[col_mun].astype(str).str.strip(),
        **({'regional': base[col_reg].astype(str).str.strip()} if col_reg else {}),
        casos_acumulados_faixas_todos_anos=base[colunas_faixa_ano].sum(axis=1),
    )
    .groupby(group_cols, as_index=False)['casos_acumulados_faixas_todos_anos']
    .sum()
)
if 'regional' not in acumulado_municipio.columns:
    acumulado_municipio['regional'] = pd.NA
if CSV_MUNICIPIOS_REGIONAIS.exists():
    regionais = pd.read_csv(CSV_MUNICIPIOS_REGIONAIS)
    regionais['codigo_municipio'] = _normalizar_codigo_municipio_mapa(regionais['codigo_ibge'])
    regionais_lookup = regionais[['codigo_municipio', 'regional_original']].rename(columns={'regional_original': 'regional_lookup'})
    acumulado_municipio = acumulado_municipio.merge(regionais_lookup, on='codigo_municipio', how='left')
    acumulado_municipio['regional'] = acumulado_municipio['regional'].fillna(acumulado_municipio['regional_lookup'])
    acumulado_municipio = acumulado_municipio.drop(columns=['regional_lookup'])
acumulado_municipio['regional'] = acumulado_municipio['regional'].fillna('—')
colunas_tabela = ['codigo_municipio', 'nome_municipio', 'regional', 'casos_acumulados_faixas_todos_anos']
print('\n' + '=' * 60)
print('🔺 10 MUNICÍPIOS COM MAIS CASOS ACUMULADOS')
print('=' * 60)
top10 = acumulado_municipio.nlargest(10, 'casos_acumulados_faixas_todos_anos')[colunas_tabela].reset_index(drop=True)
display(top10)
print('\n' + '=' * 60)
print('🔻 10 MUNICÍPIOS COM MENOS CASOS ACUMULADOS (excl. zero)')
print('=' * 60)
bottom10 = (
    acumulado_municipio[acumulado_municipio['casos_acumulados_faixas_todos_anos'] > 0]
    .nsmallest(10, 'casos_acumulados_faixas_todos_anos')[colunas_tabela]
    .reset_index(drop=True)
)
display(bottom10)
print('\n' + '=' * 60)
print(f'📊 Total geral: {acumulado_municipio["casos_acumulados_faixas_todos_anos"].sum():,.0f} casos')
print(f'   Municípios com registro: {(acumulado_municipio["casos_acumulados_faixas_todos_anos"] > 0).sum()}')
print(f'   Municípios sem registro: {(acumulado_municipio["casos_acumulados_faixas_todos_anos"] == 0).sum()}')
print('=' * 60 + '\n')
malha_mg = geobr.read_municipality(code_muni='MG', year=ANO_MALHA_MUNICIPIOS_MG_MAPA).to_crs(4326)
malha_mg['codigo_municipio'] = _normalizar_codigo_municipio_mapa(malha_mg['code_muni'])
geojson_mg = json.loads(malha_mg[['codigo_municipio', 'geometry']].to_json())
mapa_mg = malha_mg[['codigo_municipio']].merge(acumulado_municipio, on='codigo_municipio', how='left')
mapa_mg['nome_municipio'] = mapa_mg['nome_municipio'].fillna('Sem registro no CSV')
mapa_mg['casos_acumulados_faixas_todos_anos'] = pd.to_numeric(
    mapa_mg['casos_acumulados_faixas_todos_anos'], errors='coerce'
).fillna(0)
mapa_mg['regional'] = mapa_mg['regional'].fillna('—')
valores = mapa_mg['casos_acumulados_faixas_todos_anos']
limites = INTERVALOS_MANUAIS_MAPA
faixas = pd.cut(
    valores,
    bins=limites,
    include_lowest=True,
    duplicates='drop',
    right=True,
)
categorias_raw = faixas.cat.categories
rotulos = []
for iv in categorias_raw:
    lo = int(iv.left) if iv.left == int(iv.left) else iv.left
    hi = int(iv.right) if iv.right == int(iv.right) else iv.right
    rotulos.append(f'{lo:,} – {hi:,}')
faixas_rotuladas = pd.Categorical(
    [rotulos[list(categorias_raw).index(v)] if pd.notna(v) else None for v in faixas],
    categories=rotulos,
    ordered=True,
)
mapa_mg['faixa_casos'] = faixas_rotuladas.astype('object')
mapa_mg.loc[mapa_mg['casos_acumulados_faixas_todos_anos'] == 0, 'faixa_casos'] = ROTULO_SEM_CASOS
ordem_legenda = [ROTULO_SEM_CASOS] + rotulos
mapa_mg['faixa_casos'] = pd.Categorical(mapa_mg['faixa_casos'], categories=ordem_legenda, ordered=True)
cores_faixas = ['#ffffd4', '#fec44f', '#ee9c21', '#ff7f50', '#d20000', '#9c2328']
mapa_cores = {faixa: cores_faixas[i] for i, faixa in enumerate(rotulos)}
mapa_cores[ROTULO_SEM_CASOS] = COR_SEM_CASOS
hover_data = {
    'codigo_municipio': True,
    'casos_acumulados_faixas_todos_anos': ':,.0f',
    'faixa_casos': True,
    'regional': True,
}
fig = px.choropleth(
    mapa_mg,
    geojson=geojson_mg,
    locations='codigo_municipio',
    featureidkey='properties.codigo_municipio',
    color='faixa_casos',
    category_orders={'faixa_casos': ordem_legenda},
    color_discrete_map=mapa_cores,
    hover_name='nome_municipio',
    hover_data=hover_data,
    labels={
        'faixa_casos': 'Faixa de casos acumulados',
        'casos_acumulados_faixas_todos_anos': 'Casos acumulados',
        'regional': 'Regional de saúde',
    },
    title='MG — Casos acumulados por município · Todas as faixas etárias e anos',
)
fig.update_geos(
    fitbounds='locations',
    visible=False,
    showcountries=False,
    showcoastlines=False,
    showland=False,
    showframe=False,
)
fig.update_layout(
    template='plotly_white',
    width=1250,
    height=820,
    margin=dict(l=10, r=10, t=60, b=10),
    title=dict(x=0.5, xanchor='center', font_size=15),
    legend=dict(title='Casos acumulados', traceorder='normal'),
)
fig.show()
print('✅ Mapa MG com 6 faixas manuais gerado com sucesso')